# STINTSY Major Course Output

This notebook presents the development of **three supervised machine learning models** using the *Organ Retrieval and Collection of Health Information for Donation (ORCHID)* dataset.

## Objective
The primary objective is to **predict the successful transplantation of a donor heart** (Binary: *Yes/No*) based on:
- Donor demographics  
- Cause of death  
- Longitudinal clinical data, including:
  - Chemistry  
  - Complete Blood Count (CBC)  
  - Arterial Blood Gas (ABG) events  

Predict **Y** (Successful Transplant) based on **X** (Donor Profiles and Longitudinal Clinical Markers).

## Dataset
The dataset used in this project is sourced from the ORCHID database. Detailed descriptions and download links are available below:

**Link:** https://physionet.org/content/orchid/2.1.1/

## Contents of this Notebook 
- Dataset Insights
    - Unique Values
- Data Cleaning
    - Cohort Selection
    - Data Cleaning
- Exploratory Data Analysis
    - Feature distributions
    - Missing data patterns
    - Correlation analysis
    - Target vs feature analysis (IMPORTANT?)
- Data Pre-processing  
    - Feature Engineering
    - Imputation?
    - Scaling, Encoding, & Dimensionality Reduction
- Model Selection, Training, Analysis, & Tuning
- Model Evaluation and Comparison  

## Setup Instructions
Dependencies and instructions for running this notebook are provided in the following section.

# Import Libraries

In [39]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats 
import sklearn as sklearn
import torch.optim as optim
import torch.nn as nn
import torch

%matplotlib inline

# Dataset Insights

Before cleaning the data, it is essential to know the different contents of the dataset. The code below will show the different unique values in the multi-source dataset, in order to have a prior knowledge on the different features that will be relevant in completing our objective.

### OPOReferrals
This is the different donor demographics of the patients including the various outcomes of the organs in relation to transplants.

In [40]:
na_values = ["", " ", "na ", "na"]
dtype_dict = {7: str, 12: str}

OPOreferrals = pd.read_csv(
    'orchid-2.1.1/OPOReferrals.csv', 
    na_values=na_values, 
    dtype=dtype_dict,
    low_memory=False
)

cols_to_check = ['opo', 'gender', 'race', 'brain_death', 'cause_of_death_opo', 'cause_of_death_unos', 'mechanism_of_death', 
                 'circumstances_of_death', 'abo_blood_type', 'abo_rh', 'outcome_heart']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {OPOreferrals[col].nunique()}") 
    print(OPOreferrals[col].unique())
    
    null_count = OPOreferrals[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for opo ---
Count: 6
['OPO1' 'OPO2' 'OPO3' 'OPO4' 'OPO5' 'OPO6']
Missing Values (NaN): 0


--- Unique values for gender ---
Count: 2
['M' 'F' nan]
Missing Values (NaN): 61


--- Unique values for race ---
Count: 4
['White / Caucasian' 'Black / African American' 'Hispanic'
 'Other / Unknown']
Missing Values (NaN): 0


--- Unique values for brain_death ---
Count: 2
[False  True]
Missing Values (NaN): 0


--- Unique values for cause_of_death_opo ---
Count: 80
[nan 'Cardiac - Other, specify' 'ICB / ICH' 'Sepsis' 'GSW'
 'Respiratory - Other, specify' 'Pneumonia' 'Multi-system failure'
 'Drowning' 'Other, specify' 'Prematurity' 'ESRD' 'Overdose' 'SAH' 'ESLD'
 'Trauma' 'Myocardial infarction' 'AAA or thoracic AA' 'Cancer'
 'Infectious Disease - Bacterial' 'Infectious Disease - Other, specify'
 'Infectious Disease - Viral' 'Pulmonary embolism' 'Hepatitis' 'CHF' 'HIV'
 'Leukemia / Lymphoma' 'COPD' 'Arrhythmia' 'Fetal Demise' 'Exsanguination'
 'Sudden infant death syndrome' 'TR

### ABGEvents
This is the different ABG related events done to a particular patient

In [41]:
na_values = ["", " ", '""' "na ", "na"]

ABGEvents = pd.read_csv(
    'orchid-2.1.1/ABGEvents.csv', 
    na_values=na_values, 
    low_memory=False
)

cols_to_check = ['abg_ventilator_mode', 'abg_name']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {ABGEvents[col].nunique()}") 
    print(ABGEvents[col].unique())
    
    null_count = ABGEvents[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for abg_ventilator_mode ---
Count: 398
['SIMV' 'CMV' 'PRVC' 'A/C' 'Other' 'Apnea Test' 'PC' 'CPAP' 'APRV' 'NC'
 'HFOV' 'BiPAP' nan 'APVcmv' 'APV/CMV' 'PCVVG' 'SPV' 'not documented '
 'PCV' 'apv/cmv' 'AC-PC' 'APVCMV' 'P-CMV' 'spont' 'SPONT' 'VC' 'VC-AC'
 'VDR-4' '2 L/Min' '2 l/min' 'pcvvg' 'VC/AC' 'VC CMV' 'VC ' 'APNEA' 'ARPV'
 'pcv' 'PCV-VG' 'VCV' 'apnea' 'PCV-VG ' 'VDR' 'AC' 'T-Piece' 'T PIECE'
 'vc' 'PSV' 'Pressure Support' 'Apnea test' 'A/C Pressure' 'A/C-PC'
 'AC/VA' 'APV' 'Nasal Cannula' 'AC PC' 'T- Piece' 'CPAP + PS' 'AC/VC'
 'Ac-PC' 'Bagged' 'BiLevel' 'SCMV' 'T Piece' '15lpm' 'PRVC/PS'
 'SIMV(PRVC)+PS' 'PCV+' 'BVM' 'bagged' 'V/C' 'PC/AC' 'bag 15L/min'
 'apvCMV' 'ASV' 'PCV-VG+' 'nc' '.' 'apnea exam' 'ECMO' 'APV CMV' 'PC-SIMV'
 'PC- SIMV' 'PS' 'vcv' 'PCV-vg' 'nasal canula' 'Apnea ' 'na ' 'blow-by'
 'N/C' 'T-piece' 't-piece' 'VC+' 'Bilevel' 'Tpiece' 'Bipap' 'APV SIMV'
 'unknown' 'Apnea' 'PCMV' 'APV/cmv' 'VS' 'NRB' 'simv/prvc' 'Spont'
 'PRVC A/C' 'MMV' 'RA' 'nasal 

### CBCEvents
This is the different CBC related events done to a particular patient

In [42]:
na_values = ["", " ", '""' "na ", "na"]

CBCEvents = pd.read_csv(
    'orchid-2.1.1/CBCEvents.csv', 
    na_values=na_values, 
    low_memory=False
)

cols_to_check = ['cbc_name']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {CBCEvents[col].nunique()}") 
    print(CBCEvents[col].unique())
    
    null_count = CBCEvents[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for cbc_name ---
Count: 10
['WBC' 'RBC' 'Hgb' 'Hct' 'Ptl' 'Segs' 'Lymp' 'Mono' 'Eos' 'Band']
Missing Values (NaN): 0




### ChemistryEvents
This is the different chemistry related events done to a particular patient

In [43]:
na_values = ["", " ", '""' "na ", "na"]

ChemistryEvents = pd.read_csv(
    'orchid-2.1.1/ChemistryEvents.csv', 
    na_values=na_values, 
    low_memory=False
)

cols_to_check = ['chem_name']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {ChemistryEvents[col].nunique()}") 
    print(ChemistryEvents[col].unique())
    
    null_count = ChemistryEvents[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for chem_name ---
Count: 42
['Sodium' 'K' 'CI' 'CO2' 'BUN' 'Creatinine' 'CreatinineClearance'
 'Glucose' 'Calcium' 'PT' 'PTT' 'INR' 'Lactate' 'Phosphorous' 'Mg'
 'SGOTAST' 'SGPTALT' 'Albumin' 'TotalBili' 'DirectBili' 'TotalProtein'
 'AlkPhos' 'Amylase' 'Lipase' 'CKMB' 'TroponinI' 'Cpk' 'HgbA1C'
 'LipaseULN' 'Fibrinogen' 'IonizedCalcium' 'BNP' 'GGT' 'TotalMB' 'LDH'
 'SerumOsmo' 'IndirectBili' 'CpkIndex' 'TroponinT' 'SerumBetaHCG'
 'Total_CKMB_na' 'HCG']
Missing Values (NaN): 0




### CultureEvents
This is the different infection culture related events done to a particular patient

In [44]:
na_values = ["", " ", '""' "na ", "na"]

CultureEvents = pd.read_csv(
    'orchid-2.1.1/CultureEvents.csv', 
    na_values=na_values, 
    low_memory=False
)

cols_to_check = ['culture_source']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {CultureEvents[col].nunique()}") 
    print(CultureEvents[col].unique())
    
    null_count = CultureEvents[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for culture_source ---
Count: 12
['Blood' 'Sputum' 'Urine' 'Other' 'L Bronch Gm St' 'R Bronch Gm St'
 'R Bronch' 'L Bronch' 'Sputum Gm St' 'CSF' 'Wound' nan 'Lung']
Missing Values (NaN): 71




### FluidBalanceEvents
This is the different fluid balance related events done to a particular patient

In [45]:
na_values = ["", " ", '""' "na ", "na"]

FluidBalanceEvents = pd.read_csv(
    'orchid-2.1.1/FluidBalanceEvents.csv', 
    na_values=na_values, 
    low_memory=False
)

cols_to_check = ['fluid_name']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {FluidBalanceEvents[col].nunique()}") 
    print(FluidBalanceEvents[col].unique())
    
    null_count = FluidBalanceEvents[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for fluid_name ---
Count: 6
['Crystalloid' 'Colloid' 'BloodProduct' 'Urine' 'NonUrine' 'Total']
Missing Values (NaN): 0




### HemoEvents
This is the different hemo related events done to a particular patient

In [46]:
na_values = ["", " ", '""' "na ", "na"]

HemoEvents = pd.read_csv(
    'orchid-2.1.1/HemoEvents.csv', 
    na_values=na_values, 
    low_memory=False
)

cols_to_check = ['measurement_name']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {HemoEvents[col].nunique()}") 
    print(HemoEvents[col].unique())
    
    null_count = HemoEvents[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for measurement_name ---
Count: 5
['HeartRate' 'BPSystolic' 'BPDiastolic' 'UrineOutput' 'Temperature']
Missing Values (NaN): 0




### SerologyEvents
This is the different serology related events done to a particular patient

In [47]:
na_values = ["", " ", '""' "na ", "na"]

SerologyEvents = pd.read_csv(
    'orchid-2.1.1/SerologyEvents.csv', 
    na_values=na_values, 
    low_memory=False
)

cols_to_check = ['serology_name', 'serology_name_Other']

for col in cols_to_check:
    print(f"--- Unique values for {col} ---")
    print(f"Count: {SerologyEvents[col].nunique()}") 
    print(SerologyEvents[col].unique())
    
    null_count = SerologyEvents[col].isnull().sum()
    print(f"Missing Values (NaN): {null_count}")
    print("\n")

--- Unique values for serology_name ---
Count: 78
['Anti-CMV' 'Anti-HBc' 'Anti-HCV' 'Anti-HIV I/II' 'Anti-HTLV I/II'
 'Chagas' 'Chagas NAT' 'CMV IgM' 'EBNA' 'EBV (VCA) (IgG)'
 'EBV (VCA) (IgM)' 'HBcAB IgM' 'HBsAb' 'HBsAg' 'HBV NAT' 'HCV NAT'
 'HIV Ag/Ab Combo' 'HIV NAT' 'HTLV NAT' 'MHATP' 'Other1' 'Strongyloides'
 'Syphilis' 'Toxo Ab IgG' 'WNV' 'WNV NAT' 'Hepatitis BC Ab'
 'Hepatitis Bs Ag' 'Hepatitis C Ab' 'HIV 1/2 plus O Ab'
 'HIV-1/HCV/HBV NAT Ultrio' 'HTLV I/II AB' 'Anti-HBcAb'
 'HIV Ag/Ab Combo Assay' 'NAT HBV' 'NAT HCV' 'NAT HIV' 'RPR/VDRL' 'Other2'
 'Other3' 'Other4' 'Anti HBc' 'Anti HCV' 'Anti HIV 1 and 2' 'HbsAg'
 'HIV-1/HCV/HBV NAT (ULTRIO)' 'CMV' 'HBV DNA' 'HCV RNA' 'HIV-1 RNA'
 'WNV RNA' 'EBV IgG' 'EBV IgM' 'Confirmatory - Syphilis' 'Other5'
 'HBC_Total_AB' 'EBV_IgG' 'EBV_IgM' 'HCV_AB' 'HCV_NAT' 'HIV_NAT' 'RPR'
 'HIV' 'HBsAG' 'HBV_NAT' 'Toxo_IgG' 'HTLV_I_II' 'HBsAg#' 'HBc Total'
 'HCV Ab' 'HIV I/II' 'RPR#' 'HIV NAT (TMA)' 'HTLV I/II' 'ABO/Rh'
 'HCV NAT (TMA)' 'HIV O EIA' 'E

With the available datasets capturing various biological events, we can now identify which variables are most relevant to our primary objective.

In predicting the successful transplantation of a donor heart, organ viability is the key factor of interest. Each dataset (CSV file) contains clinical measurements and events that may influence this outcome. The relevant biological features extracted from these datasets are outlined below:



# 2. Data Cleaning

### 2.1 Cohort Selection
Given that this is a 100k+ rows of a dataset, we must only consider the patients that had a *outcome_heart* in the first place in order to be able to pin down what leads to a successful transplant. Counting the amount of rows that will lead to various outcomes shown below with its total count.

In [48]:
df_heart = OPOreferrals[OPOreferrals['outcome_heart'].notna()].copy()

print("--- Exact Counts for Heart Outcomes ---")
print(df_heart['outcome_heart'].value_counts())
print("\n--- Percentage Breakdown (%) ---")
print((df_heart['outcome_heart'].value_counts(normalize=True) * 100).round(2))
print(f"\nFinal Cohort Size (N): {len(df_heart)}")

--- Exact Counts for Heart Outcomes ---
outcome_heart
Transplanted                                     2987
Recovered for Research                            682
Recovered for Transplant but not Transplanted      21
Name: count, dtype: int64

--- Percentage Breakdown (%) ---
outcome_heart
Transplanted                                     80.95
Recovered for Research                           18.48
Recovered for Transplant but not Transplanted     0.57
Name: proportion, dtype: float64

Final Cohort Size (N): 3690


To formulate the heart transplant viability problem as a supervised machine learning task, the categorical **outcome_heart** column must be mapped to a binary target variable **Y** **(1/0)**.

#### Transplanted
- 1 (Success)
- The heart met all clinical criteria and was successfully utilized for transplant.
#### Recovered for Research
- 0 (Failure)
- The heart was ineligible for human transplantation due to clinical or physiological factors.
#### Recovered for Transplant but not Transplanted
- 0 (Failure)
- The heart was initially considered but ultimately rejected during the procurement process.

### 2.2 Data Cleaning

First, we need to check then clean  among the different events are medically relevant to the successfulness of a heart transplant. 

In [49]:
df_heart_cleaned = df_heart.copy()

columns_to_clean = ['opo', 'gender', 'race', 'brain_death', 'cause_of_death_opo', 'cause_of_death_unos', 
                    'mechanism_of_death', 'circumstances_of_death', 'abo_blood_type', 'abo_rh', 'outcome_heart']

# Defensive check to only clean columns that exist in the dataframe
columns_to_clean = [col for col in columns_to_clean if col in df_heart_cleaned.columns]

for col in columns_to_clean:
    df_heart_cleaned[col] = df_heart_cleaned[col].astype(str).str.lower().str.strip()

# Numeric cleaning for continuous features
numeric_cols = ['age', 'height_in', 'weight_kg']
for col in numeric_cols:
    if col in df_heart_cleaned.columns:
        df_heart_cleaned[col] = pd.to_numeric(df_heart_cleaned[col], errors='coerce')

print(f"Final Heart Cohort Size: {len(df_heart_cleaned)}")
print(f"Unique OPOs: {df_heart_cleaned['opo'].unique()}")
print(f"Unique Genders: {df_heart_cleaned['gender'].unique()}")
print(f"Unique Race Categories: {df_heart_cleaned['race'].unique()}")
print(f"Unique Brain Death: {df_heart_cleaned['brain_death'].unique()}")
print(f"Unique Causes of Death (OPO): {df_heart_cleaned['cause_of_death_opo'].unique()}")
print(f"Unique Causes of Death (UNOS): {df_heart_cleaned['cause_of_death_unos'].unique()}")
print(f"Unique Mechanisms of Death: {df_heart_cleaned['mechanism_of_death'].unique()}")
print(f"Unique Circumstances of Death: {df_heart_cleaned['circumstances_of_death'].unique()}")
print(f"Unique Blood Types: {df_heart_cleaned['abo_blood_type'].unique()}")
print(f"Unique Rh Factors: {df_heart_cleaned['abo_rh'].unique()}")
print(f"Unique Outcomes: {df_heart_cleaned['outcome_heart'].unique()}")

Final Heart Cohort Size: 3690
Unique OPOs: ['opo1' 'opo2' 'opo3' 'opo4' 'opo5' 'opo6']
Unique Genders: ['m' 'f']
Unique Race Categories: ['hispanic' 'black / african american' 'white / caucasian'
 'other / unknown']
Unique Brain Death: ['true' 'false']
Unique Causes of Death (OPO): ['nan' 'gsw' 'infectious disease - bacterial' 'drowning' 'overdose'
 'other, specify' 'icb / ich' 'sah' 'trauma'
 'respiratory - other, specify' 'pulmonary embolism'
 'cardiac - other, specify' 'myocardial infarction' 'prematurity'
 'pneumonia' 'aaa or thoracic aa' 'sudden infant death syndrome'
 'tr - gsw' 'an - asphyixiation' 'drug overdose/probable drug abuse'
 'an - other' 'car - cardiomegaly/cardiomyopathy/cardiovascular'
 'aneurysm' 'tr - mva' 'res - other' 'cva/stroke - cerebro accident'
 'other' 'intracranial hemorrhage' 'tr - other' 'an -  hanging'
 'an - drowning' 'tr - chi - closed head injury'
 'seizure/seizure disorder' 'an - smoke inhalation' 'car - probable mi'
 'res - asthma' 'esld']
Unique C

In [50]:
abg_events_cleaned = ABGEvents[ABGEvents['patient_id'].isin(df_heart_cleaned['patient_id'])].copy()
columns_to_clean = ['abg_ventilator_mode', 'abg_name']
for col in columns_to_clean:
    abg_events_cleaned[col] = abg_events_cleaned[col].astype(str).str.lower().str.strip()            

abg_events_cleaned['value'] = pd.to_numeric(abg_events_cleaned['value'], errors='coerce')

print(f"Rows in cleaned ABG: {len(abg_events_cleaned)}")
print(f"Unique Names: {abg_events_cleaned['abg_name'].unique()}")
print(f"Unique Ventilator Modes: {abg_events_cleaned['abg_ventilator_mode'].unique()}")

Rows in cleaned ABG: 566859
Unique Names: ['ph' 'pco2' 'po2' 'hco3' 'be' 'o2sat' 'fio2' 'rate' 'tv' 'peep' 'pip']
Unique Ventilator Modes: ['simv' 'other' 'a/c' 'prvc' 'aprv' 'pc' 'nc' 'apnea test' 'cmv' 'cpap'
 'hfov' 'bipap' 'spv' 'apvcmv' 'apnea' 'pcv' 'pcv-vg' 'vcv' 'ac-pc'
 't piece' 't-piece' 'bagged' 'vc' 'apv cmv' 'vc+' 'bilevel' 'psv'
 'unknown' 'prvc a/c' '8l nc o2' 'bvm' 'ac/vc' 'vc-simv' 'imv'
 'oscillator' 'apv' 'ps 5/5' 'pcvvg' 'vc-ac' 'ac-vc' 'apv/cmv' 'room air'
 'ac' 'v/c' 'simv +ps' 'asv' 'cmv a/c' 'ambu bag' 'acvc' 'simv/pc' 'n/c'
 'vent' 'pvc-vg' 'pc irv' 'simv+ps' 'pc-vg' 'not documented' 'spont'
 'simv prvc +ps' 'hfnc' 'nan' 'simv-pc' 'vc/ac' 'none' 'simv-prvc'
 'anesthesia' 'apv-simv' 'apvsimv' 'blow by' 'accmv' 'simv/ps/prvc'
 'ocilator' 'unk' 'ivr/pc' 'irv/pc' 'ippv' 'nrbmask' 'bmv' '6l nc'
 'suppled o2' '15l et' 'not listed' 'cmv/vc' 'vc ac' '10l nc' 'simv=ps'
 'apnea exam' 'vdr' 'prvc/ps' 'anesthesia bag' 'simv/prvc + ps' 'va/ac'
 'simv/prvc' 'pre apnea' 'pos

In [51]:
cbc_events_cleaned = CBCEvents[CBCEvents['patient_id'].isin(df_heart_cleaned['patient_id'])].copy()
columns_to_clean = ['cbc_name']
for col in columns_to_clean:
    cbc_events_cleaned[col] = cbc_events_cleaned[col].astype(str).str.lower().str.strip()

cbc_events_cleaned['value'] = pd.to_numeric(cbc_events_cleaned['value'], errors='coerce')

print(f"Rows in cleaned CBC: {len(cbc_events_cleaned)}")
print(f"Unique Names: {cbc_events_cleaned['cbc_name'].unique()}")

Rows in cleaned CBC: 223213
Unique Names: ['rbc' 'wbc' 'hgb' 'hct' 'ptl' 'segs' 'lymp' 'mono' 'eos' 'band']


In [52]:
chem_events_cleaned = ChemistryEvents[ChemistryEvents['patient_id'].isin(df_heart_cleaned['patient_id'])].copy()
columns_to_clean = ['chem_name', 'value_modifier']
for col in columns_to_clean:
    chem_events_cleaned[col] = chem_events_cleaned[col].astype(str).str.lower().str.strip()

chem_events_cleaned['value'] = pd.to_numeric(chem_events_cleaned['value'], errors='coerce')

print(f"Rows in cleaned Chemistry: {len(chem_events_cleaned)}")
print(f"Unique Names: {chem_events_cleaned['chem_name'].unique()}")
print(f"Unique Value Modifiers: {chem_events_cleaned['value_modifier'].unique()}")

Rows in cleaned Chemistry: 767374
Unique Names: ['sodium' 'k' 'ci' 'co2' 'bun' 'creatinine' 'creatinineclearance'
 'glucose' 'calcium' 'totalbili' 'sgotast' 'sgptalt' 'albumin'
 'totalprotein' 'alkphos' 'pt' 'ptt' 'inr' 'phosphorous' 'mg' 'ggt' 'ldh'
 'amylase' 'lipase' 'lipaseuln' 'hgba1c' 'ionizedcalcium' 'directbili'
 'lactate' 'serumosmo' 'troponini' 'cpk' 'indirectbili' 'ckmb' 'cpkindex'
 'fibrinogen' 'bnp' 'troponint' 'serumbetahcg' 'totalmb' 'total_ckmb_na'
 'hcg']
Unique Value Modifiers: ['nan' '<' '1' '0.' '0' '15' '13' '>' '1.' '<0' '4.' '4' '2.' '' '<=']


In [53]:
culture_events_cleaned = CultureEvents[CultureEvents['patient_id'].isin(df_heart_cleaned['patient_id'])].copy()

columns_to_clean = ['culture_source', 'result']

for col in columns_to_clean:
    culture_events_cleaned[col] = culture_events_cleaned[col].astype(str).str.lower().str.strip()

print(f"Rows in cleaned Culture: {len(culture_events_cleaned)}")
print(f"Unique Sources: {culture_events_cleaned['culture_source'].unique()}")
print(f"Unique Results: {culture_events_cleaned['result'].unique()}")

Rows in cleaned Culture: 18037
Unique Sources: ['blood' 'urine' 'l bronch gm st' 'r bronch gm st' 'r bronch' 'l bronch'
 'sputum gm st' 'sputum' 'other' 'csf' 'wound' 'nan' 'lung']
Unique Results: ['n' 'p' 'i']


In [54]:
fluid_balance_cleaned = FluidBalanceEvents[FluidBalanceEvents['patient_id'].isin(df_heart_cleaned['patient_id'])].copy()
columns_to_clean = ['fluid_name', 'fluid_type']
for col in columns_to_clean:
    fluid_balance_cleaned[col] = fluid_balance_cleaned[col].astype(str).str.lower().str.strip()

fluid_balance_cleaned['amount'] = pd.to_numeric(fluid_balance_cleaned['amount'], errors='coerce')

print(f"Rows in cleaned Fluid Balance: {len(fluid_balance_cleaned)}")
print(f"Unique Names: {fluid_balance_cleaned['fluid_name'].unique()}")
print(f"Unique Types: {fluid_balance_cleaned['fluid_type'].unique()}")

Rows in cleaned Fluid Balance: 30857
Unique Names: ['crystalloid' 'colloid' 'bloodproduct' 'urine' 'nonurine' 'total']
Unique Types: ['intake' 'output']


In [55]:
hemo_events_cleaned = HemoEvents[HemoEvents['patient_id'].isin(df_heart_cleaned['patient_id'])].copy()
columns_to_clean = ['measurement_name', 'measurement_type']
for col in columns_to_clean:
    hemo_events_cleaned[col] = hemo_events_cleaned[col].astype(str).str.lower().str.strip()

hemo_events_cleaned['value'] = pd.to_numeric(hemo_events_cleaned['value'], errors='coerce')

print(f"Rows in cleaned Hemodynamic Events: {len(hemo_events_cleaned)}")
print(f"Unique Names: {hemo_events_cleaned['measurement_name'].unique()}")
print(f"Unique Types: {hemo_events_cleaned['measurement_type'].unique()}")

Rows in cleaned Hemodynamic Events: 608160
Unique Names: ['heartrate' 'temperature' 'urineoutput' 'bpsystolic' 'bpdiastolic']
Unique Types: ['low' 'high' 'average' 'total']


In [56]:
serology_events_cleaned = SerologyEvents[SerologyEvents['patient_id'].isin(df_heart_cleaned['patient_id'])].copy()
columns_to_clean = ['serology_name', 'serology_name_Other', 'result']
for col in columns_to_clean: 
    serology_events_cleaned[col] = serology_events_cleaned[col].astype(str).str.lower().str.strip()

print(f"Rows in cleaned Serology: {len(serology_events_cleaned)}")
print(f"Unique Names: {serology_events_cleaned['serology_name'].unique()}")
print(f"Unique Results: {serology_events_cleaned['result'].unique()}")

Rows in cleaned Serology: 88533
Unique Names: ['anti-cmv' 'anti-hbcab' 'anti-hcv' 'anti-hiv i/ii' 'anti-htlv i/ii'
 'chagas' 'chagas nat' 'cmv igm' 'ebna' 'ebv (vca) (igg)'
 'ebv (vca) (igm)' 'hbcab igm' 'hbsab' 'hbsag' 'hbv nat' 'hcv nat'
 'hiv ag/ab combo assay' 'hiv nat' 'htlv nat' 'mhatp' 'syphilis'
 'toxo ab igg' 'wnv' 'wnv nat' 'anti-hbc' 'hiv ag/ab combo'
 'strongyloides' 'other1' 'nat hbv' 'nat hcv' 'nat hiv' 'rpr/vdrl'
 'other2' 'hepatitis bc ab' 'hepatitis bs ag' 'hepatitis c ab'
 'hiv 1/2 plus o ab' 'hiv-1/hcv/hbv nat ultrio' 'htlv i/ii ab' 'anti hbc'
 'anti hcv' 'anti hiv 1 and 2' 'ebv igg' 'hiv-1/hcv/hbv nat (ultrio)'
 'other3' 'ebv igm' 'hbv dna' 'hcv rna' 'wnv rna' 'cmv' 'hiv-1 rna'
 'other4' 'other5' 'hbc_total_ab' 'ebv_igg' 'ebv_igm' 'hcv_ab' 'hcv_nat'
 'hiv_nat' 'rpr' 'hiv' 'hbv_nat' 'toxo_igg' 'htlv_i_ii' 'hbc total'
 'hbsag#' 'hcv ab' 'hiv i/ii' 'htlv i/ii' 'rpr#' 'abo/rh' 'hcv nat (tma)'
 'hiv nat (tma)' 'hiv o eia' 'ebv']
Unique Results: ['neg' 'notdone' 'pos' 'in

Given that we already have our 3690 patients with a heart outcome, it is time to check which of these patients actually have a record for the various events that was possibly administered that will be medically relevant in determining the success of a heart transplant.

In [58]:
cohort_ids = set(df_heart_cleaned['patient_id'])

def check_coverage(cleaned_df):
    table_ids = set(cleaned_df['patient_id'])
    overlap = cohort_ids.intersection(table_ids)
    count = len(overlap)
    pct = (count / len(cohort_ids)) * 100
    return count, pct

abg_count, abg_pct = check_coverage(abg_events_cleaned)
cbc_count, cbc_pct = check_coverage(cbc_events_cleaned)
chem_count, chem_pct = check_coverage(chem_events_cleaned)
culture_count, culture_pct = check_coverage(culture_events_cleaned)
fluid_count, fluid_pct = check_coverage(fluid_balance_cleaned)
hemo_count, hemo_pct = check_coverage(hemo_events_cleaned)
serology_count, serology_pct = check_coverage(serology_events_cleaned)

print(f"--- Coverage for Cleaned Cohort (N={len(cohort_ids)}) ---")
print(f"ABG Events:             {abg_count} patients ({abg_pct:.2f}%)")
print(f"CBC Events:             {cbc_count} patients ({cbc_pct:.2f}%)")
print(f"Chemistry Events:       {chem_count} patients ({chem_pct:.2f}%)")
print(f"Culture Events:         {culture_count} patients ({culture_pct:.2f}%)")
print(f"Fluid Balance Events:   {fluid_count} patients ({fluid_pct:.2f}%)")
print(f"Hemo Events:            {hemo_count} patients ({hemo_pct:.2f}%)")
print(f"Serology Events:        {serology_count} patients ({serology_pct:.2f}%)")

--- Coverage for Cleaned Cohort (N=3690) ---
ABG Events:             3685 patients (99.86%)
CBC Events:             3685 patients (99.86%)
Chemistry Events:       3685 patients (99.86%)
Culture Events:         3035 patients (82.25%)
Fluid Balance Events:   2762 patients (74.85%)
Hemo Events:            2769 patients (75.04%)
Serology Events:        3689 patients (99.97%)


# Exploratory Data Analysis

### Feature distributions
### Missing data patterns
### Correlation analysis
### Target vs feature analysis (IMPORTANT?)

# Data Pre-processing 

### Feature Engineering
### Scaling, Encoding, & Dimensionality Reduction

# Model Selection, Training, Analysis, & Tuning

### Classical Model 1

### Classical Model 2


### Neural Network Model

# Model Evaluation

# Generative AI Use
Statement: During the preparation of this work the authors used Gemini, ChatGPT for the following purposes:
- To be able to understand medical terms 
- To be able to identify medical relationships and relevance to the objective (e.g. checking why a ventilator setting is important)


After using this tool/service, the authors reviewed and edited the content as needed and take full responsibility for the content of the publication.